# SSA vs ODE

Each experiment runs the **stochastic SSA** (Gillespie, ensemble-averaged moments)
and the **deterministic ODE** moment equations for the *same* parameters, then plots
both side by side.

**Goal:** check that the ODE = exact moment dynamics, and the SSA estimates converge
to it.

In [2]:
import sys
from pathlib import Path


sys.path.append(str(Path.cwd().parent))

%load_ext autoreload
%autoreload 2

import numpy as np

from src.ssa_simulation import simulate_telegraph, compute_sample_moments
from src.ode_moments import solve_ode_moments
from src.comparison_visualization import show_combined_moments

In [3]:
def run_comparison(title: str, n_grid=1000, time_step=None, t_end=None, **kwargs):
    base_params = {
        "k_on": 0.1,
        "k_off": 0.1,
        "k_syn": 10.0,
        "k_deg": 0.2,
        "t0": 0.0,
        "g0": 0,
        "r0": 0,
        "n_sim": 4000,
        "n_rep": 1000,
    }

    current_params = {**base_params, **kwargs}

    data = simulate_telegraph(**current_params)

    # One shared window for SSA and ODE. Default: earliest-finishing trajectory,
    # so every cell fully covers it (no held tails). Pass t_end explicitly to pin
    # a fixed window; trajectories that ended earlier then hold their last value (ZOH).
    if t_end is None:
        t_end = float(data[-1, :, 0].min())

    moments_ssa = compute_sample_moments(
        data, t_end=t_end, n_grid=n_grid, time_step=time_step
    )

    t_ode, y_ode = solve_ode_moments(
        k_on=current_params["k_on"],
        k_off=current_params["k_off"],
        k_syn=current_params["k_syn"],
        k_deg=current_params["k_deg"],
        t0=current_params["t0"],
        g0=current_params["g0"],
        r0=current_params["r0"],
        t_end=t_end,
    )

    show_combined_moments(moments_ssa, t_ode, y_ode, title=title)

### 1. Switching speed — **1a. Slow** ($k_{\text{on}}=0.01,\; k_{\text{off}}=0.02 \ll k_{\text{deg}}$)

Gene stays locked ON/OFF for long → large variance.

In [4]:
run_comparison(
    title="Experiment 1a: Slow Gene Switching Regime",
    k_on=0.01,
    k_off=0.02,
)

**1b. Fast** ($k_{\text{on}}=2,\; k_{\text{off}}=4 \gg k_{\text{deg}}$)

Promoter is effectively averaged → low variance.

In [5]:
run_comparison(
    title="Experiment 1b: Fast Gene Switching Regime",
    k_on=2,
    k_off=4,
)

### 2. Expression level — **2a. High** ($k_{\text{syn}}/k_{\text{deg}} = 40/0.5 = 80$)

High steady-state ⟨R⟩.

In [6]:
run_comparison(
    title="Experiment 2a: High Expression Regime",
    k_syn=40.0,
    k_deg=0.5,
)

**2b. Low** ($k_{\text{syn}}/k_{\text{deg}} = 2$)

Few molecules → discreteness matters; the SSA mean should still match the ODE.

In [7]:
run_comparison(
    title="Experiment 2a: High Expression Regime",
    k_syn=2.0,
    k_deg=1.0,
)

### 3. Initial conditions — **3a. Cold start** ($g_0=0,\; r_0=0$)

⟨R⟩ builds up from zero. 

In [8]:
run_comparison(
    title="Experiment 3a: Cold Start (G0=0, R0=0)",
    k_on=0.05,
    k_off=0.05,
    k_syn=10.0,
    k_deg=0.2,
    n_rep=300,
    g0=0,
    r0=0,
)

**3b. Hot start** ($g_0=1,\; r_0=50$)

⟨R⟩ relaxes down to the same steady state — ODE gives the exact relaxation curve.

In [9]:
run_comparison(
    title="Experiment 3b: Hot Start (G0=1, R0=50)",
    k_on=0.05,
    k_off=0.05,
    k_syn=10.0,
    k_deg=0.2,
    n_rep=300,
    g0=1,
    r0=50,
)

**3c. Initial mismatch** ($g_0=0,\; r_0=50$)

Gene OFF but mRNA abundant → surplus decays to equilibrium.

In [11]:
run_comparison(
    title="Experiment 3c: Initial Mismatch (G0=0, R0=50)",
    k_on=0.05,
    k_off=0.05,
    k_syn=10.0,
    k_deg=0.2,
    n_rep=300,
    g0=0,
    r0=50,
)

### 4. Numerical parameters — **4a. Ensemble size $n_{\text{rep}}$** (50 vs 1000)

ODE is the noise-free reference. 

In [12]:
run_comparison(
    title="Experiment 4a: Low Ensemble Size (n_rep=50)",
    k_on=0.05,
    n_sim=3000,
    n_rep=50,
)

In [13]:
run_comparison(
    title="Experiment 4a: High Ensemble Size (n_rep=1000)",
    k_on=0.05,
    n_sim=3000,
    n_rep=1000,
)

**4b. Run length $n_{\text{sim}}$** (400 vs 4000 events)

Sets how far in time the SSA reaches. Short runs may **stop before steady state**,
while the ODE always shows the full curve up to the same `t_end`.

In [14]:
run_comparison(
    title="Experiment 4b: Short Simulation (n_sim=400)",
    k_on=0.05,
    n_sim=400,
    n_rep=300,
)

In [15]:
run_comparison(
    title="Experiment 4b: Long Simulation (n_sim=4000)",
    k_on=0.05,
    n_sim=4000,
    n_rep=300,
)